# 01 — Exploratory Data Analysis

**Goal:** understand the shape, quality, and distributions of the raw Olist dataset before feature engineering.

Topics covered:
- Dataset overview (row counts, dtypes, null rates)
- Order volume over time
- Revenue distribution
- Customer geography
- Review score distribution
- Delivery time analysis

In [1]:
import sys
from pathlib import Path

# Make project root importable
sys.path.insert(0, str(Path.cwd().parent))

import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import duckdb

import config
from src.utils import get_duckdb_conn

In [2]:
# Load tables from DuckDB
conn = get_duckdb_conn(config.DUCKDB_PATH, read_only=True)

orders    = conn.execute("SELECT * FROM raw_orders").df()
customers = conn.execute("SELECT * FROM raw_customers").df()
payments  = conn.execute("SELECT * FROM raw_order_payments").df()
reviews   = conn.execute("SELECT * FROM raw_order_reviews").df()
items     = conn.execute("SELECT * FROM raw_order_items").df()

conn.close()

print(f"Orders:    {len(orders):>8,}")
print(f"Customers: {len(customers):>8,}")
print(f"Payments:  {len(payments):>8,}")
print(f"Reviews:   {len(reviews):>8,}")
print(f"Items:     {len(items):>8,}")

Orders:      99,441
Customers:   99,441
Payments:   103,886
Reviews:     99,224
Items:      112,650


In [3]:
# --- Order volume over time ---
orders["order_purchase_timestamp"] = pd.to_datetime(orders["order_purchase_timestamp"])
monthly = (
    orders.set_index("order_purchase_timestamp")
    .resample("ME")
    .size()
    .reset_index(name="order_count")
)

px.line(monthly, x="order_purchase_timestamp", y="order_count",
        title="Monthly Order Volume", labels={"order_purchase_timestamp": "Month"})

In [4]:
# --- Revenue distribution ---
px.histogram(payments, x="payment_value", nbins=80,
             title="Payment Value Distribution",
             labels={"payment_value": "Payment (R$)"})

In [5]:
# --- Review score distribution ---
px.histogram(reviews, x="review_score", nbins=5,
             title="Review Score Distribution",
             labels={"review_score": "Score (1–5)"})

In [6]:
# --- Purchases per customer ---
purchases_per_customer = (
    orders.merge(customers[["customer_id", "customer_unique_id"]], on="customer_id")
    .groupby("customer_unique_id")
    .size()
    .reset_index(name="n_purchases")
)

# Summary stats
print(purchases_per_customer["n_purchases"].describe().to_string())
print(f"\nCustomers with exactly 1 purchase: {(purchases_per_customer['n_purchases'] == 1).sum():,} "
      f"({(purchases_per_customer['n_purchases'] == 1).mean():.1%})")
print(f"Customers with 2+ purchases:       {(purchases_per_customer['n_purchases'] >= 2).sum():,} "
      f"({(purchases_per_customer['n_purchases'] >= 2).mean():.1%})")

# Distribution (cap at 10 for readability)
dist = (
    purchases_per_customer["n_purchases"]
    .clip(upper=10)
    .value_counts()
    .sort_index()
    .reset_index()
)
dist.columns = ["n_purchases", "n_customers"]
dist["label"] = dist["n_purchases"].astype(str).replace("10", "10+")

px.bar(dist, x="label", y="n_customers",
       title="Number of Purchases per Customer",
       labels={"label": "Purchases", "n_customers": "Customers"},
       text_auto=True)

count    96096.000000
mean         1.034809
std          0.214384
min          1.000000
25%          1.000000
50%          1.000000
75%          1.000000
max         17.000000

Customers with exactly 1 purchase: 93,099 (96.9%)
Customers with 2+ purchases:       2,997 (3.1%)
